## シグナリング理論

### 広告モデル

**プレイヤー:**
- 企業 (広告主)
- 消費者
※ 1人のプレイヤーとして扱う

**戦略:**
企業
- 高品質タイプ
- 低品質タイプ

**広告戦略:**
- 広告を出す（消費5）
- 広告を出さない（消費0）

**利得:**
企業
- 消費者が購入　→ 高品質: 10, 低品質: 10
- 消費者が購入しない → 高品質: 0, 低品質: 0
消費者
- 購入 → 高品質: 1, 低品質: -2
- 購入しない → 高品質: 0, 低品質: 0

## 状況


企業が広告を出す場合:
- 消費者は企業が高品質か分かる

企業が広告を出さない場合:
- 低品質タイプだから広告を出していない
- 高品質タイプなのに広告を出していない

## 完全ベイジアン均衡



## 資格保有のシグナリングモデル

**プレイヤー:**
- 労働者
- 企業

**労働者:**
- 高能力タイプ（資格コスト低）
- 低能力タイプ（資格コスト高）

**企業:**
- 高賃金
- 低賃金



In [8]:
import pandas as pd

# プレイヤーのタイプ（High or Low）
PType = ["H", "L"] 
# プレイーヤーの生産性
Prod = {PType[0]: 10, PType[1]: 5}

# プレイヤーのシグナル
Signal = ["Q", "N"]
# 資格取得コスト
Cost = {PType[0]: 2, PType[1]: 6}

# 企業の賃金戦略
CType = ["H", "L"]
# 賃金
Wage = {CType[0]: 8, CType[1]: 4}

df = pd.DataFrame(columns = ["Signal"] + CType)

df_H = df.copy()
for s in Signal:
    row = [s]
    for c in CType:
        if s == "Q":
            row.append((Wage[c] - Cost[PType[0]], Prod[PType[0]] - Cost[PType[0]]))
        else:
            row.append((Wage[c], Prod[PType[0]] - Cost[PType[0]]))
    df_H.loc[len(df_H)] = row

df_L = df.copy()
for s in Signal:
    row = [s]
    for c in CType:
        if s == "Q":
            row.append((Wage[c] - Cost[PType[1]], Prod[PType[1]] - Cost[PType[1]]))
        else:
            row.append((Wage[c], Prod[PType[1]] - Cost[PType[1]]))
    df_L.loc[len(df_L)] = row

display(df_H)
display(df_L)

,Signal,H,L
0,Q,"(6, 8)","(2, 8)"
1,N,"(8, 8)","(4, 8)"


,Signal,H,L
0,Q,"(2, -1)","(-2, -1)"
1,N,"(8, -1)","(4, -1)"


In [9]:
class Player:
    def __init__(self, ptype):
        self.ptype = ptype
        self.prod = Prod[ptype]
        self.cost = Cost[ptype]
        self.reset()

    def reset(self):
        self.signal = None
        self.payoff = 0

    def get_payoff(self):
        """プレイヤーの利得を計算"""
        pass

class Company:
    def __init__(self):
        # 資格の信頼度
        self.w = 0.5
        self.reset()
    
    def reset(self):
        self.payoff = 0

    def get_payoff(self):
        """企業の利得を計算"""
        pass

    def update_w(self):
        """資格の信頼度を更新"""
        pass

class Game:
    def __init__(self):
        self.player_list = []
        self.company_list = []





In [10]:
import random
import pandas as pd

# 労働者のタイプ
PType = ["H", "L"]

# タイプごとの生産性
Prod = {
    "H": 10,
    "L": 5
}

# シグナル
# Q: 資格あり, N: 資格なし
Signal = ["Q", "N"]

# タイプごとの資格取得コスト
Cost = {
    "H": 2,
    "L": 6
}

# 企業の行動と賃金
# Reject は不採用を表す
Wage = {
    "HighWage": 8,
    "LowWage": 4,
    "Reject": 0
}


class Player:
    def __init__(self, ptype):
        # 労働者のタイプ H or L
        self.ptype = ptype

        # 労働者の生産性
        self.prod = Prod[ptype]

        # 選択したシグナル Q or N
        self.signal = None

        # 労働者の利得
        self.payoff = 0

    def reset(self):
        """労働者の状態を初期化する"""
        self.signal = None
        self.payoff = 0

    def choose_signal(self, company, epsilon=0.0):
        """
        企業の現在の反応を見て、資格を取るかどうかを選ぶ

        epsilon:
            ランダムに行動する確率。
            これを少し入れると、学習・探索っぽい挙動になる。
        """

        # 一定確率でランダムにシグナルを選ぶ
        if random.random() < epsilon:
            self.signal = random.choice(Signal)
            return self.signal

        # 資格あり Q を選んだ場合の企業行動を予想
        action_Q = company.decide_action("Q")

        # 資格なし N を選んだ場合の企業行動を予想
        action_N = company.decide_action("N")

        # 資格ありの場合の労働者利得
        payoff_Q = Wage[action_Q] - Cost[self.ptype]

        # 資格なしの場合の労働者利得
        payoff_N = Wage[action_N]

        # 利得が高い方のシグナルを選ぶ
        if payoff_Q > payoff_N:
            self.signal = "Q"
        elif payoff_Q < payoff_N:
            self.signal = "N"
        else:
            # 同じ利得ならランダムに選ぶ
            self.signal = random.choice(Signal)

        return self.signal

    def get_payoff(self, action):
        """
        労働者の利得を計算する

        労働者利得 = 賃金 - 資格取得コスト
        """

        # 企業が提示した賃金
        wage = Wage[action]

        # 資格ありならコストが発生し、資格なしならコスト0
        signal_cost = Cost[self.ptype] if self.signal == "Q" else 0

        # 労働者の利得を計算
        self.payoff = wage - signal_cost

        return self.payoff


class Company:
    def __init__(self, belief_Q=0.7, belief_N=0.3, threshold=0.6, alpha=0.1):
        """
        belief_Q:
            企業が「資格あり Q の人はHタイプである」と信じる確率

        belief_N:
            企業が「資格なし N の人はHタイプである」と信じる確率

        threshold:
            高賃金を出すための信念の基準

        alpha:
            信念更新の学習率
        """

        # シグナルごとの信念
        self.belief = {
            "Q": belief_Q,
            "N": belief_N
        }

        # 高賃金を出すかどうかの基準
        self.threshold = threshold

        # 信念更新の速さ
        self.alpha = alpha

        # 企業の利得
        self.payoff = 0

    def reset(self):
        """企業の期ごとの利得を初期化する"""
        self.payoff = 0

    def expected_prod(self, signal):
        """
        企業がシグナルを見たときの期待生産性を計算する

        期待生産性 = P(H|signal) * Hの生産性 + P(L|signal) * Lの生産性
        """

        # シグナルに対する信念
        mu = self.belief[signal]

        # 期待生産性を計算
        exp_prod = mu * Prod["H"] + (1 - mu) * Prod["L"]

        return exp_prod

    def decide_action(self, signal):
        """
        企業がシグナルを見て、賃金・採用を決める

        belief が高ければ HighWage
        belief が低ければ LowWage
        期待生産性が低すぎれば Reject
        """

        # シグナルから期待生産性を計算
        exp_prod = self.expected_prod(signal)

        # 低賃金を払っても期待利得がマイナスなら不採用
        if exp_prod < Wage["LowWage"]:
            return "Reject"

        # Hタイプである信念が十分高ければ高賃金
        if self.belief[signal] >= self.threshold:
            return "HighWage"

        # それ以外は低賃金
        return "LowWage"

    def get_payoff(self, player, action):
        """
        企業の利得を計算する

        企業利得 = 労働者の生産性 - 賃金
        """

        # 不採用なら企業利得は0
        if action == "Reject":
            profit = 0
        else:
            profit = player.prod - Wage[action]

        # 企業の累積利得に加える
        self.payoff += profit

        return profit

    def update_belief(self, hired_players):
        """
        採用後の成果を見て、企業の信念を更新する

        ここでは簡単化して、
        採用した労働者の本当のタイプを企業が後から観察できる
        と仮定する。
        """

        # QとNそれぞれについて信念を更新する
        for signal in Signal:

            # そのシグナルを出して採用された労働者だけを取り出す
            observed = [
                p for p in hired_players
                if p.signal == signal
            ]

            # 観察データがない場合は信念を更新しない
            if len(observed) == 0:
                continue

            # 観察されたHタイプの割合を計算
            observed_H_rate = sum(p.ptype == "H" for p in observed) / len(observed)

            # 古い信念と観察結果を混ぜて更新する
            self.belief[signal] = (
                (1 - self.alpha) * self.belief[signal]
                + self.alpha * observed_H_rate
            )


class Game:
    def __init__(self, n_players=100, p_H=0.5, T=50, epsilon=0.02):
        """
        n_players:
            1期あたりの労働者数

        p_H:
            Hタイプの割合

        T:
            シミュレーション期間

        epsilon:
            労働者がランダムに行動する確率
        """

        # 1期あたりの労働者数
        self.n_players = n_players

        # Hタイプの割合
        self.p_H = p_H

        # シミュレーション期間
        self.T = T

        # ランダム探索の確率
        self.epsilon = epsilon

        # 企業を作成
        self.company = Company()

        # 期ごとの結果を保存するリスト
        self.history = []

    def make_player(self):
        """HタイプまたはLタイプの労働者を生成する"""

        # 確率p_HでHタイプ、それ以外はLタイプ
        ptype = "H" if random.random() < self.p_H else "L"

        return Player(ptype)

    def run_one_period(self, t):
        """1期分のゲームを実行する"""

        # 労働者を生成
        players = [
            self.make_player()
            for _ in range(self.n_players)
        ]

        # 採用された労働者を保存するリスト
        hired_players = []

        # 個人単位の結果を保存するリスト
        period_rows = []

        # 企業の期ごとの利得を初期化
        self.company.reset()

        # 各労働者についてゲームを実行
        for player in players:

            # 労働者が資格あり/なしを選ぶ
            signal = player.choose_signal(
                self.company,
                epsilon=self.epsilon
            )

            # 企業がシグナルを見て行動を決める
            action = self.company.decide_action(signal)

            # 労働者の利得を計算
            worker_payoff = player.get_payoff(action)

            # 企業の利得を計算
            firm_payoff = self.company.get_payoff(player, action)

            # 不採用でなければ、採用者リストに追加
            if action != "Reject":
                hired_players.append(player)

            # 1人分の結果を保存
            period_rows.append({
                "t": t,
                "ptype": player.ptype,
                "signal": signal,
                "action": action,
                "worker_payoff": worker_payoff,
                "firm_payoff": firm_payoff,
                "belief_Q_before_update": self.company.belief["Q"],
                "belief_N_before_update": self.company.belief["N"]
            })

        # 採用後の成果を見て、企業の信念を更新
        self.company.update_belief(hired_players)

        # DataFrameに変換
        df_period = pd.DataFrame(period_rows)

        # 期ごとの集計結果を保存
        summary = {
            "t": t,
            "belief_Q": self.company.belief["Q"],
            "belief_N": self.company.belief["N"],
            "Q_rate_H": df_period.query("ptype == 'H'")["signal"].eq("Q").mean(),
            "Q_rate_L": df_period.query("ptype == 'L'")["signal"].eq("Q").mean(),
            "avg_worker_payoff": df_period["worker_payoff"].mean(),
            "avg_firm_payoff": df_period["firm_payoff"].mean()
        }

        # 履歴に追加
        self.history.append(summary)

        return df_period

    def run(self):
        """シミュレーション全体を実行する"""

        # 全期間の個人単位データを保存するリスト
        all_rows = []

        # T期分繰り返す
        for t in range(self.T):

            # 1期分のゲームを実行
            df_period = self.run_one_period(t)

            # 個人単位データを保存
            all_rows.append(df_period)

        # 全期間の個人単位データ
        result_df = pd.concat(all_rows, ignore_index=True)

        # 期ごとの集計データ
        history_df = pd.DataFrame(self.history)

        return result_df, history_df
    
# 乱数を固定する
random.seed(0)

# ゲームを作成する
game = Game(
    n_players=100,
    p_H=0.5,
    T=50,
    epsilon=0.02
)

# シミュレーションを実行する
result_df, history_df = game.run()

# 期ごとの集計結果を確認する
history_df.head()

# 最後の数期間を見る
history_df.tail()

,t,belief_Q,belief_N,Q_rate_H,Q_rate_L,avg_worker_payoff,avg_firm_payoff
45,45,0.983302,0.013519,0.980000,0.020000,4.96,1.50
46,46,0.983247,0.012167,1.000000,0.023256,5.12,1.53
47,47,0.984923,0.010950,1.000000,0.000000,5.02,1.51
48,48,0.984612,0.012078,0.981818,0.022222,5.06,1.55
49,49,0.984110,0.012831,0.979592,0.019608,4.94,1.49
